# Random Forest Regression
- ensemble model that uses multiple trees to make more stable predictions

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

#import from the data_splitter Python script to load and split the data - prevents having to copy and paste code
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..'))) #used to move to the parent folder
from src.data_splitter import load_data, split

#load dataset and variables
df, X, y = load_data()

#split data by groups
X_train, X_val, X_test, y_train, y_val, y_test = split(df, X, y)

Training: 10661 rows | 70 unique players
Validation: 2169 rows | 15 unique players
Testing: 2170 rows | 15 unique players


In [21]:
#need to use one hot encoding to encode movement_type because it's a string - models can only work with numerical values
categorical = ["movement_type"]
numeric = [col for col in X_train.columns if col not in categorical]

#preprocess data: since it's a tree, don't need to scale numerical features but categorical features need to be converted to numeric using OneHotEncoding
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)
    ],
    remainder="passthrough" #keeps but doesn't change numerical info
)

#combine preprocessor and random forest regression in a pipeline (does it in sequential steps)
rf_pipeline0 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            oob_score=True
        ))
    ]
)

rf_pipeline0.fit(X_train, y_train)
print("Random forest pipeline (without tuning) fitted successfully on training data")

Random forest pipeline (without tuning) fitted successfully on training data


In [22]:
#evaluation on random forest regressor using validation data
#predict target values for unseen validation data
y_val_pred = rf_pipeline0.predict(X_val)

#calculate regression metrics: mean absolute error, root mean squared error, R^2
val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = root_mean_squared_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

#performance summary
print("=== RANDOM FOREST REGRESSOR MODEL 0 METRICS (NO TUNING) (Validation Set) ===")
print(f"MAE  : {val_mae:.4f}")
print(f"RMSE : {val_rmse:.4f}")
print(f"R²   : {val_r2:.4f}")

=== RANDOM FOREST REGRESSOR MODEL 0 METRICS (NO TUNING) (Validation Set) ===
MAE  : 3.4478
RMSE : 4.3417
R²   : 0.6120
